### Phase 1: Data Audit

In [2]:
import pandas as pd

# Carregar ficheiro
df = pd.read_csv('../Datasets/dados_streaming.csv')
df.head()

,id_cliente,idade,mensalidade,meses_contrato,suporte_tecnico,target
0,1000,56.0,17.60,12,Não,Fiel
1,1001,69.0,12.06,44,Não,Fiel
2,1002,46.0,16.13,36,Não,Cancelou
3,1003,32.0,10.75,4,Não,Cancelou
4,1004,60.0,10.42,5,Não,Fiel


In [3]:
# how many customers are missing the age
missing_age = df['idade'].isnull().sum()
print(f'Existem {missing_age} idades em falta')

# Fill the values with the median age
median_age = df['idade'].mean()
df['idade'] = df['idade'].fillna(median_age)

print(f'Agora existem {df['idade'].isnull().sum()} idades em falta')

Existem 15 idades em falta
Agora existem 0 idades em falta


In [4]:
# how many unique values are in column suporte_tecnico
unique_values = len(df['suporte_tecnico'].unique())
print(f'Existem {unique_values} valores únicos na coluna suporte_tecnico')

# how many unique values are in target
unique_target = len(df['target'].unique())
print(f'Existem {unique_target} valores únicos na coluna target')

Existem 2 valores únicos na coluna suporte_tecnico
Existem 2 valores únicos na coluna target


In [5]:
proportions = df['target'].value_counts(normalize=True)
percent_cancelled = proportions['Cancelou'] * 100

print(f'Existem {percent_cancelled:.0f}% clientes que cancelaram o servico')

Existem 29% clientes que cancelaram o servico


### Phase 2: Filters and Logic

In [6]:
# novo dataset com clientes com mensalidade superior 
# a 15.00 e com contrato superior a 12 meses
clientes_premium = df[(df['mensalidade'] > 15.00) & (df['meses_contrato'] > 12)]
clientes_premium.head()

,id_cliente,idade,mensalidade,meses_contrato,suporte_tecnico,target
2,1002,46.000000,16.13,36,Não,Cancelou
5,1005,25.000000,15.21,37,Sim,Fiel
7,1007,56.000000,16.27,42,Sim,Fiel
8,1008,46.195876,17.16,28,Sim,Fiel
9,1009,40.000000,19.66,31,Sim,Fiel


In [7]:
# remover a coluna id_cliente
df = df.drop('id_cliente', axis=1)
df.head()

,idade,mensalidade,meses_contrato,suporte_tecnico,target
0,56.0,17.60,12,Não,Fiel
1,69.0,12.06,44,Não,Fiel
2,46.0,16.13,36,Não,Cancelou
3,32.0,10.75,4,Não,Cancelou
4,60.0,10.42,5,Não,Fiel


### Phase 3: Model Preparation (Encoding)

In [8]:
mapping = {'Fiel': 0, 'Cancelou': 1}
df['target'] = df['target'].map(mapping)
print(df['target'].head())

0    0
1    0
2    1
3    1
4    0
Name: target, dtype: int64


In [9]:
mapping = {'Sim': 1, 'Não': 0}
df['suporte_tecnico'] = df['suporte_tecnico'].map(mapping)
print(df['suporte_tecnico'].head())

0    0
1    0
2    0
3    0
4    0
Name: suporte_tecnico, dtype: int64


### Phase 4: ML Insights

In [10]:
average_contrato_canceled = df[df['target'] == 1]['meses_contrato'].mean()
average_contrato_loyal = df[df['target'] == 0]['meses_contrato'].mean()
print(f"Média Cancelados: {average_contrato_canceled:.2f}")
print(f"Média Fiéis: {average_contrato_loyal:.2f}")

Média Cancelados: 25.43
Média Fiéis: 24.25


In [11]:
correlacao = df['mensalidade'].corr(df['target'])

print(f"A correlação entre mensalidade e cancelamento é: {correlacao:.4f}")

A correlação entre mensalidade e cancelamento é: -0.0805


### Extra

In [13]:
import numpy as np

data = df['mensalidade'] * df['meses_contrato']
df['Total_amount'] = data

mapping = {0: 'Normal', 1: 'VIP'}
# np.where: se a condição for verdadeira, atribui 1, caso contrário 0
df['Customer_Class'] = np.where(df['Total_amount'] > 500, 1, 0)
df['Customer_Class'] = df['Customer_Class'].map(mapping)
df.head()

,idade,mensalidade,meses_contrato,suporte_tecnico,target,Total_amount,Customer_Class
0,56.0,17.60,12,0,0,211.20,Normal
1,69.0,12.06,44,0,0,530.64,VIP
2,46.0,16.13,36,0,1,580.68,VIP
3,32.0,10.75,4,0,1,43.00,Normal
4,60.0,10.42,5,0,0,52.10,Normal
